<a href="https://colab.research.google.com/github/IntanFatiha29/Tubes_AIoT_Intan-Fatiha_201012520033/blob/main/TUBES_AIoT_Intan_Fatiha_201012520033.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === 1. IMPORT LIBRARY ===

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import requests
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

In [23]:
# "=== 2. MEMBUAT DATASET ==="
n = 300
time_index = np.arange(n)

# Lux dibuat 0 - 5000 agar kategori gelap, sedang, terik muncul
lux = 2500 + 2000 * np.sin(2 * np.pi * time_index / 24)
temperature = 28 + 4 * np.sin(2 * np.pi * (time_index - 6) / 24)
humidity = 60 - 10 * np.sin(2 * np.pi * (time_index - 6) / 24)

# Noise sensor
lux_noise = np.random.normal(0, 250, n)
temp_noise = np.random.normal(0, 1.2, n)
hum_noise = np.random.normal(0, 4, n)

lux = lux + lux_noise
temperature = temperature + temp_noise
humidity = humidity + hum_noise

# Batas nilai sensor
lux = np.clip(lux, 0, 5000)
temperature = np.clip(temperature, 15, 45)
humidity = np.clip(humidity, 20, 100)

# Label posisi tirai sesuai TA
shade_position = []

for l in lux:
    if l > 3000:
        shade_position.append(0)      # Terik = tertutup
    elif l >= 1000:
        shade_position.append(45)     # Sedang = setengah
    else:
        shade_position.append(90)     # Gelap = terbuka

df = pd.DataFrame({
    "lux": lux,
    "temperature": temperature,
    "humidity": humidity,
    "shade_position": shade_position
})

display(df.head())

,lux,temperature,humidity,shade_position
0,2624.178538,23.005206,73.027954,45
1,2983.072015,23.464079,65.970597,45
2,3661.922135,25.432651,72.138678,0
3,4294.971026,25.904017,72.493619,0
4,4173.512464,25.974918,66.653740,0


In [24]:
# Simpan dataset
df.to_csv('aiot_smart_curtain_dataset.csv', index=False)

print("Dataset berhasil disimpan")
display(df.head())

Dataset berhasil disimpan


,lux,temperature,humidity,shade_position
0,2624.178538,23.005206,73.027954,45
1,2983.072015,23.464079,65.970597,45
2,3661.922135,25.432651,72.138678,0
3,4294.971026,25.904017,72.493619,0
4,4173.512464,25.974918,66.653740,0


In [25]:
# === 3. MEMBUAT TARGET PREDIKSI 1 JAM KE DEPAN ===

df_model = df.copy()

df_model["lux_next"] = df_model["lux"].shift(-1)
df_model["temperature_next"] = df_model["temperature"].shift(-1)
df_model["humidity_next"] = df_model["humidity"].shift(-1)

df_model = df_model.dropna()

# Simpan dataset ML
df_model.to_csv("window_shade_ml_dataset.csv", index=False)

print("Dataset ML berhasil disimpan sebagai window_shade_ml_dataset.csv")
display(df_model.head())

Dataset ML berhasil disimpan sebagai window_shade_ml_dataset.csv


,lux,temperature,humidity,shade_position,lux_next,temperature_next,humidity_next
0,2624.178538,23.005206,73.027954,45,2983.072015,23.464079,65.970597
1,2983.072015,23.464079,65.970597,45,3661.922135,25.432651,72.138678
2,3661.922135,25.432651,72.138678,0,4294.971026,25.904017,72.493619
3,4294.971026,25.904017,72.493619,0,4173.512464,25.974918,66.653740
4,4173.512464,25.974918,66.653740,0,4373.317413,27.105517,70.095374


In [26]:
# === 4. MEMBUAT FITUR LAG ===

df_model["lux_lag1"] = df_model["lux"].shift(1)
df_model["temp_lag1"] = df_model["temperature"].shift(1)
df_model["hum_lag1"] = df_model["humidity"].shift(1)

df_model["lux_lag2"] = df_model["lux"].shift(2)
df_model["temp_lag2"] = df_model["temperature"].shift(2)
df_model["hum_lag2"] = df_model["humidity"].shift(2)

df_model = df_model.dropna()

X = df_model[[
    "lux", "temperature", "humidity",
    "lux_lag1", "temp_lag1", "hum_lag1",
    "lux_lag2", "temp_lag2", "hum_lag2"
]]

y = df_model[[
    "lux_next",
    "temperature_next",
    "humidity_next"
]]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

X shape: (297, 9)
y shape: (297, 3)


,lux,temperature,humidity,lux_lag1,temp_lag1,hum_lag1,lux_lag2,temp_lag2,hum_lag2
2,3661.922135,25.432651,72.138678,2983.072015,23.464079,65.970597,2624.178538,23.005206,73.027954
3,4294.971026,25.904017,72.493619,3661.922135,25.432651,72.138678,2983.072015,23.464079,65.970597
4,4173.512464,25.974918,66.653740,4294.971026,25.904017,72.493619,3661.922135,25.432651,72.138678
5,4373.317413,27.105517,70.095374,4173.512464,25.974918,66.653740,4294.971026,25.904017,72.493619
6,4894.803204,29.533198,56.904843,4373.317413,27.105517,70.095374,4173.512464,25.974918,66.653740


In [27]:
# === 5. SPLIT DATA TRAINING DAN TESTING ===

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Jumlah data training:", len(X_train))
print("Jumlah data testing :", len(X_test))

Jumlah data training: 237
Jumlah data testing : 60


In [28]:
# === 6. TRAINING RANDOM FOREST REGRESSOR ===

rf_reg = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_split=3,
    min_samples_leaf=1,
    random_state=42
)

rf_reg.fit(X_train, y_train)

print("Training Random Forest selesai")

Training Random Forest selesai


In [29]:
# === 7. EVALUASI MODEL ===

y_pred = rf_reg.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 80.53180294075561
RMSE: 177.7787913067269
R2  : 0.8553992878996567


In [30]:
# === 8. SIMPAN MODEL ===

joblib.dump(rf_reg, "rf_regressor_smart_curtain.pkl")

print("Model berhasil disimpan sebagai rf_regressor_smart_curtain.pkl")

Model berhasil disimpan sebagai rf_regressor_smart_curtain.pkl


In [31]:
# === 9. FUNGSI KEPUTUSAN TIRAI SESUAI THRESHOLD TA ===

def decide_curtain(lux_next):
    if lux_next > 3000:
        return 0      # Terik = tirai tertutup
    elif lux_next >= 1000:
        return 45     # Sedang = tirai setengah terbuka
    else:
        return 90     # Gelap = tirai terbuka penuh

In [32]:
# === 10. TEST PREDIKSI MANUAL ===

sample = pd.DataFrame([{
    "lux": 3500,
    "temperature": 30,
    "humidity": 60,
    "lux_lag1": 3400,
    "temp_lag1": 29,
    "hum_lag1": 61,
    "lux_lag2": 3300,
    "temp_lag2": 28,
    "hum_lag2": 62
}])

pred = rf_reg.predict(sample)[0]

lux_next = pred[0]
temp_next = pred[1]
hum_next = pred[2]

position = decide_curtain(lux_next)

print("Lux_next        :", round(lux_next, 1))
print("Temperature_next:", round(temp_next, 1))
print("Humidity_next   :", round(hum_next, 1))
print("Posisi Tirai    :", position, "derajat")

Lux_next        : 3649.0
Temperature_next: 29.3
Humidity_next   : 55.9
Posisi Tirai    : 0 derajat


In [39]:
# === 11. KONEKSI FIREBASE + ML  ===

FIREBASE_URL = "https://aiot-smartcurtain-default-rtdb.asia-southeast1.firebasedatabase.app"

last_lux = 0
last_temp = 0
last_hum = 0

while True:
    sensor_url = FIREBASE_URL + "/sensor.json"
    sensor_data = requests.get(sensor_url).json()

    if sensor_data is None:
        print("Sensor belum tersedia")
        time.sleep(5)
        continue

    lux = float(sensor_data["lux"])
    temperature = float(sensor_data["temperature"])
    humidity = float(sensor_data["humidity"])

    X_input = pd.DataFrame([{
        "lux": lux,
        "temperature": temperature,
        "humidity": humidity,
        "lux_lag1": last_lux,
        "temp_lag1": last_temp,
        "hum_lag1": last_hum,
        "lux_lag2": last_lux,
        "temp_lag2": last_temp,
        "hum_lag2": last_hum
    }])

    pred = rf_reg.predict(X_input)[0]

    lux_next = float(pred[0])
    temp_next = float(pred[1])
    hum_next = float(pred[2])

    position = decide_curtain(lux_next)

    result = {
        "prediction": {
            "lux_next": round(lux_next, 1),
            "temperature_next": round(temp_next, 1),
            "humidity_next": round(hum_next, 1)
        },
        "curtain": {
            "position": position
        }
    }

    requests.patch(FIREBASE_URL + "/.json", json=result)

    print("Sensor:", sensor_data)
    print("Prediction:", result)
    print("Data ML terkirim ke Firebase")
    print("-" * 40)

    last_lux = lux
    last_temp = temperature
    last_hum = humidity

    time.sleep(5)

Sensor: {'humidity': 81.5, 'lux': 2976.6, 'temperature': 68.1}
Prediction: {'prediction': {'lux_next': 2804.1, 'temperature_next': 27.2, 'humidity_next': 61.2}, 'curtain': {'position': 45}}
Data ML terkirim ke Firebase
----------------------------------------
Sensor: {'humidity': 81.5, 'lux': 2976.6, 'temperature': 68.1}
Prediction: {'prediction': {'lux_next': 2838.2, 'temperature_next': 29.7, 'humidity_next': 54.1}, 'curtain': {'position': 45}}
Data ML terkirim ke Firebase
----------------------------------------
Sensor: {'humidity': 81.5, 'lux': 2976.6, 'temperature': 68.1}
Prediction: {'prediction': {'lux_next': 2838.2, 'temperature_next': 29.7, 'humidity_next': 54.1}, 'curtain': {'position': 45}}
Data ML terkirim ke Firebase
----------------------------------------
Sensor: {'humidity': 81.5, 'lux': 2976.6, 'temperature': 68.1}
Prediction: {'prediction': {'lux_next': 2838.2, 'temperature_next': 29.7, 'humidity_next': 54.1}, 'curtain': {'position': 45}}
Data ML terkirim ke Firebase
-

KeyboardInterrupt: 